FETCHING LANDING DATA SAVE FOLDER LOCATION

In [0]:
version=dbutils.jobs.taskValues.get(
    taskKey="SNAPSHOT_VERSION_FETCH",
    key="SNAPSHOT_VERSION",
    debugValue=""
    )

new_landing_data_dir=f"/Volumes/pep_dev/raw/landing/{version}"

display(new_landing_data_dir)

In [0]:
if not version:
    raise ValueError("No snapshot version from SNAPSHOT_VERSION_FETCH; stopping before load and delete.")

In [0]:
# df=(spark.readStream.format("cloudFiles")
#     .option("cloudFiles.Format","csv")
#     # .option("cloudFiles.includeExistingFiles", "true")
#     .option("cloudFiles.schemaLocation","/Volumes/pep_dev/raw/landing/20260916065435-fec/")
#     .option("header", "true")
#     .load("/Volumes/pep_dev/raw/landing/20260916065435-fec/")
#     )



In [0]:
from pyspark.sql.functions import input_file_name, regexp_extract, current_timestamp

In [0]:
# dbutils.fs.rm(f"{CHECKPOINT}/stream", recurse=True)
# dbutils.fs.rm(f"{CHECKPOINT}/schema", recurse=True)

In [0]:
# %sql
# DELETE FROM pep_dev.bronze.statements
# WHERE _snapshot_version IN ('20260914005301-omh','20260916065435-fec');

In [0]:
from pyspark.sql.functions import col, regexp_extract, current_timestamp

LANDING="/Volumes/pep_dev/raw/landing"
CHECKPOINT="/Volumes/pep_dev/bronze/_checkpoints"


df=(spark.readStream.format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("cloudFiles.schemaLocation",f"{CHECKPOINT}/schema")
    .option("header", "true")
     .option("multiLine", "true")
    .option("escape", '"')
    .load(LANDING)
    .withColumn("_snapshot_version",regexp_extract(col("_metadata.file_path"),r"/landing/([^/]+)/", 1))
    .withColumn("_ingestion_ts",current_timestamp())
          )



In [0]:
(df.writeStream
.format("delta")
.option("checkPointLocation",f"{CHECKPOINT}/stream")
.option("mergeSchema","True")
.outputMode("append")
.trigger(availableNow=True)
.toTable("pep_dev.bronze.statements")
.awaitTermination()
)

In [0]:

spark.sql("""INSERT INTO pep_dev.bronze.snapshot_version_control
(snapshot_version,created_at)
SELECT :version,current_timestamp()
    WHERE NOT EXISTS
        (SELECT 1 FROM pep_dev.bronze.snapshot_version_control 
        WHERE snapshot_version=:version)
        """,args={"version":version}
        )




In [0]:
df=(spark.read.format("delta")
.option("header", "true")
.option("multiLine", "true")
.option("escape", '"')
.table("pep_dev.bronze.statements"))

display(df.count())



In [0]:
df_detail = spark.sql("DESCRIBE DETAIL pep_dev.bronze.statements")

# Extract the bytes and convert to GB
bytes_size = df_detail.select("sizeInBytes").collect()[0][0]
gb_size = bytes_size / (1024 ** 3)

print(f"Table Size: {gb_size:.2f} GB")

In [0]:
%sql
-- SELECT COUNT(*),_snapshot_version FROM pep_dev.bronze.statements GROUP BY _snapshot_version;

In [0]:
dbutils.fs.rm(new_landing_data_dir, recurse=True)